In [ ]:
import pandas as pd
import numpy as np
import torch

from config import Config
from data import load_transfer_pair
from train import train_one_pair, set_seed

In [ ]:
import torch
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")

In [ ]:
cfg = Config()

In [ ]:
SRC = "restaurant"
TGT = "laptop"

(
src_train,
src_val,
tgt_train,
tgt_test,
word2id,
pos2id,
emb
) = load_transfer_pair(
    SRC,
    TGT
)

print(len(src_train))
print(len(src_val))
print(len(tgt_train))
print(len(tgt_test))

In [ ]:
scores = []

for seed in cfg.seeds:

    print("="*60)
    print("SEED:", seed)

    set_seed(seed)

    p,r,f1 = train_one_pair(
        src_train,
        src_val,
        tgt_train,
        tgt_test,
        word2id,
        pos2id,
        emb,
        run_name=f"{SRC}_{TGT}_{seed}"
    )

    print(
        f"Target F1 = {f1:.4f}"
    )

    scores.append(f1)

print("="*60)
print(
    "AVG F1 =",
    round(np.mean(scores)*100,2)
)
print(
    "STD =",
    round(np.std(scores)*100,2)
)


In [ ]:
import shutil
import os

seed = 1

src = SRC
tgt = TGT

old_path = f"./saved_models/{src}_{tgt}_{seed}.pt"
new_path = f"./saved_models/BEST_{src}_TO_{tgt}.pt"

shutil.copy(
    old_path,
    new_path
)

print("saved:", new_path)

In [ ]:
rows = []

for src,tgt in cfg.transfer_pairs:

    print("\n" + "="*70)
    print(src, "->", tgt)

    (
    src_train,
    src_val,
    tgt_train,
    tgt_test,
    word2id,
    pos2id,
    emb
    ) = load_transfer_pair(
        src,
        tgt
    )

    pair_scores = []

    for seed in cfg.seeds:

        set_seed(seed)

        p,r,f1 = train_one_pair(
            src_train,
            src_val,
            tgt_train,
            tgt_test,
            word2id,
            pos2id,
            emb,
            run_name=f"{src}_{tgt}_{seed}"
        )

        pair_scores.append(f1)

    mean_f1 = np.mean(pair_scores)
    std_f1  = np.std(pair_scores)

    rows.append([
        src,
        tgt,
        mean_f1,
        std_f1
    ])

In [ ]:
df = pd.DataFrame(
    rows,
    columns=[
        "Source",
        "Target",
        "Mean_F1",
        "Std"
    ]
)

df["Mean_F1"] = (
    df["Mean_F1"] * 100
).round(2)

df["Std"] = (
    df["Std"] * 100
).round(2)

display(df)

print(
    "Overall Avg =",
    round(
        df["Mean_F1"].mean(),
        2
    )
)
